I am part of an institution.   
I initially don't know how the institution is organized  
Every day, I can decide whether to code, write or design.   
By taking action over time, I learn more about how the institution is organized.   
This helps me infer whether I should be a coder, writer or designer in this institution.  

I can start by modelling a single agent interacting with the institution and figuring out their role over time. 

Once I have this single agent model, I can then move to a multi agent model where the institution itself is made up of the agents. 

In [1]:
# Import dependencies
from functools import cache
import jax
import jax.numpy as np
from memo import memo, domain
from enum import IntEnum
import matplotlib.pyplot as plt

In [29]:
class ACTIONS(IntEnum):
    CODE = 0 
    WRITE = 1
    DESIGN = 2

# Instituitional Performance at each step. 
STATES = np.arange(1,11) 

# History of actions and instituitional performance at each step
# For simplicity, just 2 rounds of history (i.e. 3 rounds of game)
HISTORY = domain(
    s0 = len(STATES),
    a0 = len(ACTIONS),
    s1 = len(STATES),
    a1 = len(ACTIONS)
    )

@jax.jit
def Tr(t, h, state_, action_, h_):
    # Update h to h_ with state_ and action_ at time t 
    z = HISTORY._tuple(h)
    z = np.array(z)
    z = z.at[t * 2].set(state_)
    z = z.at[t * 2 + 1].set(action_)
    return h_ == H(*z)

@jax.jit
def is_init(h):
    return h == 0

@jax.jit
def reward(state, action):
    return state + action - 1

# discount factor
@jax.jit
def gamma():
    return 1.0

In [31]:
# Need some mechanism to update the history!

@memo 
def Q[history: HISTORY, state: STATES, action: ACTIONS](t):
    alice: knows(state, action)
    
    # Start with history at round t -1 
    alice: chooses(history in HISTORY, wpp=Q[history, state, action](t-1) if t > 1 else is_init(history))

    # Alice makes a move 
    alice: given(state_ in STATES, wpp=Tr(state, action, state_))
    alice: chooses(action_ in ACTIONS, to_maximize = 0.0 if t < 0 else Q[history, state_, action_](t-1))

    # History gets updated 
    alice: chooses(h_ in H, wpp=Tr(t-1, history, state_, action_, h_))
    
    
    # alice: knows(history, state, action)
    # alice: given(state_ in STATES, wpp=Tr(state, action, state_))
    # alice: chooses(action_ in ACTIONS, to_maximize = 0.0 if t < 0 else Q[history, state_, action_](t-1))
    # alice: chooses(history in HISTORY, )
    return 1

In [28]:
# F = np.arange(2)
# F
len(F)



# @jax.jit
# def Tr(r, h, fi, ft, h_):
#     # step game: update h to h_ with moves (fi, ft) at round r
#     z = H._tuple(h)
#     z = np.array(z)
#     z = z.at[r * 2].set(fi)
#     z = z.at[r * 2 + 1].set(ft)
#     return h_ == H(*z)

# HISTORY
z = HISTORY._tuple(1)
z = np.array(z)
z.at[20].set(5*3)
z

2